In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F             
import math

class InputEmbedding(nn.Module):
    def __init__(self, vocab_size: int, d_model: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)

In [15]:
d_model = 8
vocab_size = 1000
embed = InputEmbedding(vocab_size, d_model)
sentence_tokens = torch.tensor([[10,25,500,30,31,85]])
output = embed(sentence_tokens)
print(output.shape)

torch.Size([1, 6, 8])


In [16]:
print(output)

tensor([[[-1.5606,  3.4120, -0.1041,  1.0405,  3.3091,  1.8203,  1.3191,
          -2.3123],
         [-0.5937, -2.2034,  2.6866, -3.3918, -0.9320,  1.6157, -1.9577,
           1.5983],
         [-0.0379, -1.3446,  2.3832,  3.0575,  0.9039,  0.2719,  0.2831,
           1.4498],
         [ 6.1806,  2.4111, -1.0022, -2.0716, -2.8307, -0.6361,  2.7767,
          -3.2622],
         [-2.3810, -2.7628,  0.4058,  1.4672,  2.4158,  0.2588,  1.9938,
          -0.0416],
         [ 5.7435, -0.9061, -1.1885,  1.7642, -2.5932, -1.8495,  2.0595,
           2.6396]]], grad_fn=<MulBackward0>)


In [19]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, seq_len: int , dropout: float ):
        super().__init__()
        self.d_model = d_model
        self.dropout = nn.Dropout(p=dropout)
        self.seq_len = seq_len
        pe = torch.zeros(seq_len, d_model)
        position = torch.arange(0, seq_len  , dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)
        return self.dropout(x)

In [20]:
d_model = 8
seq_len = 1000
pos_enc = PositionalEncoding(d_model, seq_len, dropout=0.1)
x = embed(sentence_tokens)
output = pos_enc(x)
print(output.shape)

torch.Size([1, 6, 8])


In [21]:
print(output)

tensor([[[-1.7340,  4.9022, -0.1156,  0.0000,  3.6768,  3.1337,  1.4657,
          -1.4581],
         [ 0.2753, -1.8478,  3.0961, -2.6631, -1.0244,  2.9063, -2.1741,
           0.0000],
         [ 0.9682, -1.9564,  0.0000,  4.4861,  1.0265,  1.4130,  0.0000,
           2.7220],
         [ 7.0241,  1.5790, -0.7852, -1.2403, -3.1119,  0.4038,  3.0885,
          -2.5136],
         [-3.4865, -3.7961,  0.8836,  2.6536,  2.7287,  1.3978,  2.2197,
           1.0648],
         [ 5.3162, -0.6916, -0.7878,  2.9353, -2.8259, -0.9453,  2.2939,
           4.0439]]], grad_fn=<MulBackward0>)


In [22]:
class MultiHeadAttention(nn.Module):

    def __init__(self, h: int, d_model: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.h = h
        assert d_model % h == 0, "d_model must be divisible by h"
        self.d_k = d_model // h
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        attention_probs = F.softmax(attention_scores, dim=-1)
        if dropout is not None:
            attention_probs = dropout(attention_scores)
        return (attention_scores @ value, attention_probs)
    
    def forward(self, q, k, v, mask):
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)
        
        query = query.view(query[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key = key.view(key[0], key.shape[1], self.h, self.d_k).transpose(1, 2)
        value = value.view(value[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        x, self.attention_scores = MultiHeadAttention.attention(query, key, value, mask, self.dropout)
        return self.w_o(x)